# Mission 2 — 신고자(1) / 119대원(0) 발화 분류: 학습 기록

**최종 모델**: 통화 단위 트랜스포머 10개의 확률 평균 (`mission2_final10.ckpt`). 서울 Validation 발화 111,947개 Accuracy **0.9277**.

| 단계 | 코드 | 내용 |
| --- | --- | --- |
| 1. 특징 캐시 | `training_code/make_feature_cache.py` (Colab GPU) | 발화마다 `startAt`~`endAt`을 잘라 ECAPA 화자 임베딩 192차원(`speechbrain/spkrec-ecapa-voxceleb`, 가중치 고정)과 MFCC 통계 261차원을 저장 |
| 2. 학습 | `training_code/train_call_model.py` (CPU) | 통화의 발화 전체를 한 시퀀스로 넣는 트랜스포머(위치 인코딩 없음) 10개를 설정만 바꿔 학습 |
| 3. 묶기 | `training_code/pack_model.py` | 10개 가중치 + ECAPA 인코더 파일을 체크포인트 한 파일로 |
| 추론 | `inference.py` (+ `features.py`, `model.py`) | 공식 명령 한 번으로 CSV 생성 |

**데이터·annotation 사용**
- 학습: 서울 Training 29,200통화만 사용. Validation 3,640통화는 학습(역전파)과 epoch 선택에 쓰지 않았다. epoch 선택은 Training 통화 10% holdout으로 했다.
- 입력: annotation에서는 `startAt`, `endAt`(발화 자르기)만 쓴다. `text`와 발화 순서는 쓰지 않는다. `speaker`는 학습 정답으로만 쓴다.
- 모델에 위치 인코딩이 없어 발화를 어떤 차례로 넣어도 결과가 같다(발화 순서를 쓸 수 없는 구조).
- 사전학습 모델은 공개 모델(ECAPA)만 사용했고 상용 AI API는 쓰지 않는다.

**이 노트북에 대해**: 10개를 노트북 안에서 다시 학습하면 CPU로 약 13시간이 걸려 다시 돌리지 않았다. 대신 학습 코드로
1 epoch를 실제로 다시 실행해 원래 로그와 같은 값이 나오는지 확인하고(2절), 원래 학습 실행의 로그 파일을 그대로 보여 주며(3절),
저장된 결과와 제출 체크포인트로 정확도를 다시 계산한다(4~5절).

In [1]:
import sys, json, time, subprocess, argparse, os
from pathlib import Path
import numpy as np

SUBMISSION = Path.cwd()                              # 이 노트북이 있는 제출 폴더
TRAINING = SUBMISSION / "training_code"
DATA = Path(r"C:/유니짜장/DCC_Aideas/mission2_local") # 특징 캐시와 원래 학습 실행 결과가 있는 폴더
RUNS = DATA / "runs"                                 # 원래 학습 실행: 로그(.log), 결과(.json), Validation 확률(.npy)
WORK = DATA / "work_notebook"                        # 이 노트북 전용 작업 폴더
sys.path[:0] = [str(SUBMISSION), str(TRAINING)]
from train_call_model import load_split, make_call
from model import CallEnsemble, predict_calls
FINAL = ["A0_base", "A1_seed7", "A3_seed11", "F0_full", "A2_wide",
         "C0_pair", "C1_pair_seed7", "C2_pair_w1", "C3_pair_seed11", "F1_full_pair"]
caches = argparse.Namespace(embed_cache_dir=DATA / "mission2_embed_cache", mfcc_cache_dir=DATA / "mission2_cache", work_dir=WORK)
train = load_split(caches, "train")
val = load_split(caches, "val")
print("캐시 정렬 확인 완료 (ECAPA와 MFCC의 정답·통화 경계가 같음)")

캐시 정렬 확인 완료 (ECAPA와 MFCC의 정답·통화 경계가 같음)


## 1. 데이터 분석
발화 길이는 MFCC 특징의 마지막 값(log(1 + 길이 초))에서 되돌린다. 정답(speaker)은 분석과 학습 정답에만 쓴다.

In [2]:
def describe(name, data):
    labels, counts = data["labels"], data["counts"]
    seconds = np.expm1(np.asarray(data["mfcc"][:, -1], dtype=np.float64))
    call_id = np.repeat(np.arange(len(counts)), counts)
    caller_share = (np.bincount(call_id, weights=labels, minlength=len(counts)) / np.maximum(counts, 1))[counts > 0]
    print(f"[{name}] 통화 {len(counts):,} | 발화 {len(labels):,} | 신고자(1) {labels.mean():.1%} / 119대원(0) {1 - labels.mean():.1%}")
    print(f"  통화당 발화 수: 중앙값 {np.median(counts):.0f}, 평균 {counts.mean():.1f}, 90% {np.percentile(counts, 90):.0f}, 최대 {counts.max()}")
    print(f"  통화 안 신고자 발화 비율: 중앙값 {np.median(caller_share):.2f} (한 화자만 있는 통화 {np.mean((caller_share == 0) | (caller_share == 1)):.1%})")
    for value, who in ((0, "119대원"), (1, "신고자")):
        s = seconds[labels == value]
        print(f"  {who} 발화 길이(초): 중앙값 {np.median(s):.2f}, 평균 {s.mean():.2f}, 1초 미만 {np.mean(s < 1):.1%}")
describe("Training", train)
describe("Validation", val)

[Training] 통화 29,200 | 발화 889,358 | 신고자(1) 52.0% / 119대원(0) 48.0%
  통화당 발화 수: 중앙값 29, 평균 30.5, 90% 46, 최대 98
  통화 안 신고자 발화 비율: 중앙값 0.52 (한 화자만 있는 통화 0.0%)
  119대원 발화 길이(초): 중앙값 1.40, 평균 1.82, 1초 미만 30.8%
  신고자 발화 길이(초): 중앙값 1.49, 평균 2.12, 1초 미만 33.5%
[Validation] 통화 3,640 | 발화 111,947 | 신고자(1) 51.9% / 119대원(0) 48.1%
  통화당 발화 수: 중앙값 29, 평균 30.8, 90% 47, 최대 95
  통화 안 신고자 발화 비율: 중앙값 0.52 (한 화자만 있는 통화 0.0%)
  119대원 발화 길이(초): 중앙값 1.40, 평균 1.82, 1초 미만 31.2%
  신고자 발화 길이(초): 중앙값 1.48, 평균 2.12, 1초 미만 33.6%


## 2. 학습 코드 실행 확인 (A0_base 설정으로 1 epoch 재실행)
최종 모델 A0_base와 같은 설정(seed 42, 20 epoch 스케줄, CPU 14스레드)으로 첫 epoch만 실제로 다시 돌린다.
같은 코드라면 원래 학습 로그의 첫 epoch 줄과 loss·holdout 정확도가 같아야 한다.

In [3]:
start = time.time()
result = subprocess.run([sys.executable, str(TRAINING / "train_call_model.py"),
                         "--embed_cache_dir", str(caches.embed_cache_dir), "--mfcc_cache_dir", str(caches.mfcc_cache_dir),
                         "--work_dir", str(WORK / "rerun"), "--out", str(WORK / "rerun" / "A0_epoch1.pt"),
                         "--epochs", "20", "--run_epochs", "1", "--threads", "14"],
                        capture_output=True, text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
rerun = [line for line in result.stdout.splitlines() if "Warning" not in line]
print("\n".join(rerun) or result.stderr[-3000:])
print(f"소요 {time.time() - start:.0f}초\n")
original = next(line for line in (RUNS / "A0_base.log").read_text(encoding="utf-8").splitlines() if line.startswith("[epoch  1]"))
again = next(line for line in rerun if line.startswith("[epoch  1]"))
print("원래 로그:", original)
print("재실행  :", again)
print("loss·holdout 정확도 일치:", original.split(" (")[0] == again.split(" (")[0])

캐시 로딩: Train 발화 889,358 / 통화 29,200, Validation 발화 111,947 / 통화 3,640 (0시간 0분 19초)
모델: {'in_dim': 906, 'd_model': 128, 'layers': 3, 'heads': 4, 'ff': 256, 'dropout': 0.1}, 파라미터 514,177개, 장치 cpu
[epoch  1] loss 0.2590, Train holdout Accuracy 0.9080 ← 최고 (0시간 7분 20초)
최고 epoch 1: Train holdout Accuracy 0.9080
Validation Accuracy (트랜스포머): 0.9047
  119대원(0) recall 0.8519
  신고자(1) recall 0.9537
저장: C:\유니짜장\DCC_Aideas\mission2_local\work_notebook\rerun\A0_epoch1.pt / 전체 소요 0시간 7분 57초
소요 491초

원래 로그: [epoch  1] loss 0.2590, Train holdout Accuracy 0.9080 ← 최고 (0시간 5분 49초)
재실행  : [epoch  1] loss 0.2590, Train holdout Accuracy 0.9080 ← 최고 (0시간 7분 20초)
loss·holdout 정확도 일치: True


## 3. 최종 10개 학습 로그 (원래 실행 로그 파일)
로컬 PC(CPU 16코어)에서 `train_call_model.py`로 학습했다(모델당 1~2시간, 합계 약 13시간). 아래는 실행 때 저장된 로그 파일을 그대로 읽은 것이다.
Train 전체 학습 2개(F0, F1)는 holdout 실험에서 정한 20 epoch를 그대로 학습하고 마지막 epoch를 썼다.

In [4]:
sweep = {json.loads(l)["name"]: json.loads(l) for l in (RUNS / "sweep.jsonl").read_text(encoding="utf-8").splitlines()}
for name in FINAL:
    print("=" * 30, name, "| 추가 인자:", sweep[name]["args"].replace("--epochs 20 --threads 14", "").strip() or "(기본)")
    for line in (RUNS / f"{name}.log").read_text(encoding="utf-8").splitlines():
        if line.startswith(("모델:", "[epoch", "holdout이", "최고 epoch", "Validation Accuracy (트랜스포머)", "  119대원", "  신고자")):
            print(line)

============================== A0_base | 추가 인자: (기본)
모델: {'in_dim': 906, 'd_model': 128, 'layers': 3, 'heads': 4, 'ff': 256, 'dropout': 0.1}, 파라미터 514,177개, 장치 cpu
[epoch  1] loss 0.2590, Train holdout Accuracy 0.9080 ← 최고 (0시간 5분 49초)
[epoch  2] loss 0.1918, Train holdout Accuracy 0.9133 ← 최고 (0시간 4분 47초)
[epoch  3] loss 0.1819, Train holdout Accuracy 0.9143 ← 최고 (0시간 4분 56초)
[epoch  4] loss 0.1760, Train holdout Accuracy 0.9172 ← 최고 (0시간 4분 58초)
[epoch  5] loss 0.1715, Train holdout Accuracy 0.9188 ← 최고 (0시간 5분 19초)
[epoch  6] loss 0.1673, Train holdout Accuracy 0.9191 ← 최고 (0시간 4분 58초)
[epoch  7] loss 0.1648, Train holdout Accuracy 0.9205 ← 최고 (0시간 4분 47초)
[epoch  8] loss 0.1616, Train holdout Accuracy 0.9215 ← 최고 (0시간 2분 56초)
[epoch  9] loss 0.1577, Train holdout Accuracy 0.9225 ← 최고 (0시간 3분 22초)
[epoch 10] loss 0.1545, Train holdout Accuracy 0.9235 ← 최고 (0시간 3분 31초)
[epoch 11] loss 0.1517, Train holdout Accuracy 0.9242 ← 최고 (0시간 2분 28초)
[epoch 12] loss 0.1484, Train holdout Accura

## 4. 앙상블 정확도 (학습 때 저장한 Validation 확률)
섞는 규칙은 미리 정했다: 트랜스포머끼리 가중치 없이 단순 평균, 0.5 이상이면 신고자(1).

In [5]:
from sklearn.metrics import classification_report
labels = val["labels"]
probs = {name: np.load(RUNS / f"{name}.val_probability.npy") for name in FINAL}
acc = lambda p: float(((p >= .5) == labels).mean())
for name in FINAL:
    print(f"{name:<16} 단독 Validation {acc(probs[name]):.4f}")
final = np.mean([probs[n] for n in FINAL], axis=0)
print(f"\n3개 평균 (A0·A1·A3): {acc(np.mean([probs[n] for n in FINAL[:3]], axis=0)):.4f}")
print(f"최종 10개 평균:      {acc(final):.4f}\n")
print(classification_report(labels, (final >= .5).astype(int), target_names=["119대원(0)", "신고자(1)"], digits=4))

A0_base          단독 Validation 0.9219
A1_seed7         단독 Validation 0.9226
A3_seed11        단독 Validation 0.9240
F0_full          단독 Validation 0.9233
A2_wide          단독 Validation 0.9234
C0_pair          단독 Validation 0.9241
C1_pair_seed7    단독 Validation 0.9244
C2_pair_w1       단독 Validation 0.9236
C3_pair_seed11   단독 Validation 0.9237
F1_full_pair     단독 Validation 0.9252

3개 평균 (A0·A1·A3): 0.9263
최종 10개 평균:      0.9277



              precision    recall  f1-score   support

    119대원(0)     0.9472    0.8999    0.9229     53845
      신고자(1)     0.9113    0.9535    0.9319     58102

    accuracy                         0.9277    111947
   macro avg     0.9293    0.9267    0.9274    111947
weighted avg     0.9286    0.9277    0.9276    111947



## 5. 제출 체크포인트 재현 확인
`pack_model.py`로 묶은 제출 체크포인트를 추론 코드와 같은 모델 코드(`model.py`)로 불러 Validation 캐시에서 다시 계산한다.
4절의 학습 결과와 같아야 한다.

In [6]:
import joblib
checkpoint = joblib.load(SUBMISSION / "mission2_final10.ckpt")
print("체크포인트:", checkpoint["model_type"], "| 트랜스포머", len(checkpoint["transformers"]), "개:",
      ", ".join(entry["source"] for entry in checkpoint["transformers"]))
ensemble = CallEnsemble(checkpoint["transformers"])
outputs = []
for model, scale in ensemble.models:
    inputs = [make_call(val, call, scale) for call in range(len(val["counts"]))]
    outputs.append(np.concatenate(predict_calls(model, inputs)))
reproduced = np.mean(outputs, axis=0)
print(f"체크포인트로 다시 계산한 Validation 정확도: {acc(reproduced):.4f}")
print(f"학습 때 확률과 최대 차이: {np.abs(reproduced - final).max():.2e}, 판정이 다른 발화: {int(((reproduced >= .5) != (final >= .5)).sum())}")

체크포인트: mission2_call_transformer | 트랜스포머 10 개: A0_base.pt, A1_seed7.pt, A3_seed11.pt, F0_full.pt, A2_wide.pt, C0_pair.pt, C1_pair_seed7.pt, C2_pair_w1.pt, C3_pair_seed11.pt, F1_full_pair.pt


체크포인트로 다시 계산한 Validation 정확도: 0.9277
학습 때 확률과 최대 차이: 1.16e-07, 판정이 다른 발화: 0


## 6. 실데이터 추론 검증 (Colab T4 GPU, 공식 명령)

```
python inference.py --audio_dir <Validation WAV> --label_dir <Validation JSON> --ckpt_path mission2_final10.ckpt --output ./outputs/mission2.csv
```

| 항목 | 결과 |
| --- | --- |
| 입력 | Validation 통화 3,640개 WAV·JSON (WAV를 새로 읽어 특징을 처음부터 계산) |
| 출력 | `audio file name,startAt,endAt,speaker` 헤더, 111,947행 |
| **Accuracy** | **0.9277** (4절 학습 결과와 같음) |
| 추론 시간 | 1시간 16분 (Drive에서 WAV 읽기 포함) |
| 환경 | Python 3.13, torch 2.11.0+cu130, speechbrain 1.1.1, librosa 0.11.0, numpy 2.1.3 (`requirements.txt`로 설치) |

## 7. 오류 분석 (최종 10개 평균, Validation)

In [7]:
seconds = np.expm1(np.asarray(val["mfcc"][:, -1], dtype=np.float64))
correct = (final >= .5) == labels
print("발화 길이별 정확도")
for low, high in ((0, 0.5), (0.5, 1), (1, 2), (2, 4), (4, 8), (8, 1e9)):
    mask = (seconds >= low) & (seconds < high)
    print(f"  {low:>4}~{'' if high > 1e8 else high:<4}초: 발화 {mask.sum():>6,} ({mask.mean():5.1%}), 정확도 {correct[mask].mean():.4f}")
size = np.repeat(val["counts"], val["counts"])
print("통화 발화 수별 정확도")
for low, high in ((1, 6), (6, 16), (16, 31), (31, 61), (61, 10_000)):
    mask = (size >= low) & (size < high)
    print(f"  {low:>3}~{high - 1:<4}개: 발화 {mask.sum():>6,} ({mask.mean():5.1%}), 정확도 {correct[mask].mean():.4f}")
call_id = np.repeat(np.arange(len(val["counts"])), val["counts"])
call_accuracy = (np.bincount(call_id, weights=correct, minlength=len(val["counts"])) / np.maximum(val["counts"], 1))[val["counts"] > 0]
print(f"통화별 정확도: 전부 맞힌 통화 {np.mean(call_accuracy == 1):.1%}, 90% 이상 {np.mean(call_accuracy >= .9):.1%}, 절반 이하 {np.mean(call_accuracy <= .5):.1%}")
confident = np.abs(final - .5) >= .4
print(f"확신 높은 발화(확률 0.1 이하·0.9 이상) {confident.mean():.1%}: 정확도 {correct[confident].mean():.4f} / 나머지 정확도 {correct[~confident].mean():.4f}")

발화 길이별 정확도
     0~0.5 초: 발화  8,954 ( 8.0%), 정확도 0.9358
   0.5~1   초: 발화 27,392 (24.5%), 정확도 0.9016
     1~2   초: 발화 36,843 (32.9%), 정확도 0.9053
     2~4   초: 발화 26,384 (23.6%), 정확도 0.9565
     4~8   초: 발화 11,230 (10.0%), 정확도 0.9846
     8~    초: 발화  1,144 ( 1.0%), 정확도 0.9895
통화 발화 수별 정확도
    1~5   개: 발화     13 ( 0.0%), 정확도 1.0000
    6~15  개: 발화  2,106 ( 1.9%), 정확도 0.9539
   16~30  개: 발화 45,142 (40.3%), 정확도 0.9346
   31~60  개: 발화 60,409 (54.0%), 정확도 0.9231
   61~9999개: 발화  4,277 ( 3.8%), 정확도 0.9074
통화별 정확도: 전부 맞힌 통화 19.0%, 90% 이상 76.2%, 절반 이하 0.0%
확신 높은 발화(확률 0.1 이하·0.9 이상) 81.0%: 정확도 0.9889 / 나머지 정확도 0.6667


## 결과 요약 (서울 Validation 발화 111,947개)

| 단계 | Accuracy |
| --- | ---: |
| MFCC 통계 + RandomForest (발화 하나만) | 0.8372 |
| ECAPA 임베딩 + 통화 내 비교 2단계 | 0.9009 |
| 같은 화자 판별기 2단계 (트랜스포머 이전 최고) | 0.9117 |
| 통화 단위 트랜스포머 단독 | 0.9219 ~ 0.9252 |
| **최종: 트랜스포머 10개 평균** | **0.9277** |